# Weather Pipeline — Analysis

Queries the raw partitioned Parquet data using DuckDB and produces exploratory plots.

**Prerequisites:** Run the Airflow DAG at least once so `data/weather_partitioned/` is populated.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/weather_partitioned")
PARQUET_GLOB = str(DATA_DIR / "**/*.parquet")

con = duckdb.connect()
con.execute(f"CREATE VIEW weather AS SELECT * FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=true)")
print("Columns:", con.execute("DESCRIBE weather").fetchdf()["column_name"].tolist())

## 1. Row counts per city

In [ ]:
counts = con.execute("""
    SELECT city, COUNT(*) AS days
    FROM weather
    GROUP BY city
    ORDER BY days DESC
""").fetchdf()

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(counts["city"], counts["days"])
ax.set_xlabel("Days of data")
ax.set_title("Records per city")
plt.tight_layout()
plt.show()
counts

## 2. Monthly average max temperature by city

In [ ]:
monthly = con.execute("""
    SELECT city, year, month,
           AVG(tempmax) AS avg_tempmax,
           AVG(tempmin) AS avg_tempmin
    FROM weather
    GROUP BY city, year, month
    ORDER BY city, year, month
""").fetchdf()

monthly["period"] = pd.to_datetime(
    monthly[["year", "month"]].assign(day=1)
)

fig, ax = plt.subplots(figsize=(14, 6))
for city, grp in monthly.groupby("city"):
    ax.plot(grp["period"], grp["avg_tempmax"], marker="o", markersize=3, label=city)

ax.set_xlabel("Month")
ax.set_ylabel("Avg max temperature (°C)")
ax.set_title("Monthly avg max temperature per city")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Hottest and coldest days overall

In [ ]:
extremes = con.execute("""
    SELECT city, datetime, tempmax, tempmin
    FROM weather
    ORDER BY tempmax DESC
    LIMIT 10
""").fetchdf()

print("Top 10 hottest days:")
display(extremes)

coldest = con.execute("""
    SELECT city, datetime, tempmax, tempmin
    FROM weather
    ORDER BY tempmin ASC
    LIMIT 10
""").fetchdf()

print("Top 10 coldest days:")
display(coldest)

## 4. Temperature range (max - min) by city

In [ ]:
ranges = con.execute("""
    SELECT city,
           MAX(tempmax) AS all_time_max,
           MIN(tempmin) AS all_time_min,
           MAX(tempmax) - MIN(tempmin) AS temp_range
    FROM weather
    GROUP BY city
    ORDER BY temp_range DESC
""").fetchdf()

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(ranges))
ax.bar(x, ranges["all_time_max"], label="All-time max", alpha=0.7)
ax.bar(x, ranges["all_time_min"], label="All-time min", alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(ranges["city"], rotation=45, ha="right")
ax.set_ylabel("Temperature (°C)")
ax.set_title("All-time temperature range by city")
ax.legend()
plt.tight_layout()
plt.show()
ranges